# ② Hán OCR + consensus (Qwen-VL) + bge-m3 alignment — Minh Mệnh Chính Yếu  ·  notebook 2 / 2

Hán side + alignment, in a **clean paddle env** (no Surya). Reuses notebook ① 's
`out/<VOL>/` (split manifest, rendered Hán pages, `vi_sentences.jsonl`).

Runs **build_dicts → step 3 (Hán OCR) → step 3b (consensus clean) → step 4
(bge-m3 alignment)**. Step 3b re-reads each Hán column with **Qwen2.5-VL** and
fixes PaddleOCR mistakes BEFORE alignment, so the bge dense+sparse signals see
corrected characters (method validated in notebook 4; logic in
`pipeline/han_consensus.py`). Set `USE_CONSENSUS=False` to fall back to raw OCR.

**Runs ALL 6 volumes in one loop.** Each volume is processed end-to-end and its
result zip is pushed to Drive **as soon as that volume finishes**. A volume that
fails is logged and skipped. Run notebook ① for every VOL first (`out_zips/<VOL>.zip`).

> **GPU memory:** Qwen-7B (4-bit, ~6 GB) co-resides with bge-m3 — comfortable on
> L4/A100, tight on a T4. If a T4 OOMs: keep `vlm_mode="cascade"`, or set
> `USE_CONSENSUS=False`.

## 1. Dependencies (paddle + bge stack — NO surya)  →  then **Runtime ▸ Restart**

In [1]:
# Hán OCR (paddle) + consensus (Qwen2.5-VL) + alignment (bge-m3). Surya intentionally absent.
!pip -q install PyMuPDF opencv-python-headless pandas openpyxl tqdm huggingface_hub
!pip -q install FlagEmbedding                   # bge-m3 alignment
# consensus arbiter: Qwen2.5-VL + helpers (opencc folds traditional/simplified for voting)
!pip -q install "transformers>=4.49" qwen-vl-utils accelerate bitsandbytes opencc-python-reimplemented
!pip install paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu129/
!pip install -U "paddleocr[doc-parser]>=3.4.0"
# REPAIR torch NCCL — MUST be last (paddle-gpu downgrades it, breaks `import torch`).
!pip install -U nvidia-nccl-cu12
print('\n>>> Now: Runtime ▸ Restart session, then run VERIFY.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.2 MB/s eta 0:00:00
Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu129/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 GB 448.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 16.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 15.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.4/781.4 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 580.7/580.7 MB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

### Verify paddle + torch GPU (after restart)

In [1]:
import numpy, paddle, torch
print('numpy', numpy.__version__, '| paddle', paddle.__version__, '| torch', torch.__version__)
assert paddle.device.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0, \
    'paddle GPU not visible — Runtime ▸ GPU, reinstall, restart'
assert torch.cuda.is_available(), 'torch CUDA broken — re-run nvidia-nccl-cu12, restart'
paddle.set_device('gpu')
print('paddle GPU conv OK:', tuple(paddle.nn.Conv2D(3,8,3)(paddle.randn([1,3,32,32])).shape))
print('torch CUDA OK:', torch.zeros(1).cuda().device)

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


numpy 2.0.2 | paddle 3.3.1 | torch 2.11.0+cu128
paddle GPU conv OK: (1, 8, 30, 30)
torch CUDA OK: cuda:0


## 2. Mount Drive + project

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/minh_menh'
MMCY  = '/content/mmcy'                       # LOCAL work root (same as notebook 1)
os.environ['MMCY_ROOT'] = MMCY
%cd $DRIVE
!mkdir -p {MMCY}/out
!rsync -a {DRIVE}/assets/ {MMCY}/assets/      # dicts (local, for build_dicts + step3/4)
print('code:', DRIVE, '| work root (local):', MMCY)

Mounted at /content/drive
/content/drive/MyDrive/minh_menh
code: /content/drive/MyDrive/minh_menh | work root (local): /content/mmcy


## 4. Build dictionaries (one-time; downloads Unihan ~8 MB)
Volume-independent. **Must precede step 3.**

In [4]:
!python -m pipeline.build_dicts

[03:46:37] build_dicts INFO: loaded 21 manual hanviet overrides
[03:46:37] build_dicts INFO: loaded 12 curated hanviet supplements
[03:46:37] build_dicts INFO: kVietnamese readings: 8318 chars (21 overridden)
[03:46:37] build_dicts INFO: KanjiDictVN readings: 9991 chars
[03:46:37] build_dicts INFO: variant candidates: 15565 chars
[03:46:38] build_dicts INFO: wrote hanviet.csv (18099 chars: 8318 kVietnamese, 6582 KanjiDictVN, 7 supplement, 3192 teacher, 0 variant)
[03:46:38] build_dicts INFO: wrote QuocNgu_SinoNom.dic (5371 entries)
[03:46:38] build_dicts INFO: wrote SinoNom_Similar.dic (15590 entries)
[03:46:38] build_dicts INFO: done. S1=15590  S2=5371  hanviet=18099


## 3. Run ALL 6 volumes — per-vol push to Drive

Edit `VOLS` below to run a subset. The BGE model **and** the Qwen-VL arbiter load
**once** and are reused for every volume. For each `VOL` the loop: unzips NB1's
hand-off → step3 Hán OCR → **step3b consensus clean (Qwen-VL fixes characters)** →
step4 bge-m3 alignment on the corrected text (→ `output_bge/<VOL>/`) → char/review
lanes + box overlays → stages one folder → **zips & uploads `output/<VOL>.zip`**.

In [ ]:
# ── Setup for the 6-vol loop: load BGE model ONCE, define align helpers ──
from pathlib import Path
import numpy as np, pandas as pd
from FlagEmbedding import BGEM3FlagModel
from tqdm.auto import tqdm
from pipeline import config
from pipeline.common import read_jsonl, write_jsonl
from pipeline.step4_align import filter_vi_sentences, lexical_sim

VOLS = ["vol1", "vol2", "vol3", "vol4", "vol5", "vol6"]   # edit to run a subset

ALIGN = dict(config.ALIGN)
BGE_SIM_THRESHOLD = 0.50      # bge cosines ~0.45-0.66 (median ~0.57), not LaBSE's ~0-0.35
BGE_SUSPECT_DENSE = 0.45
ALIGN["sim_threshold"] = BGE_SIM_THRESHOLD
BATCH = 256

model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)   # load ONCE, reuse every vol

def _merge_sparse(weights, idxs):
    m = {}
    for k in idxs:
        for t, w in weights[k].items(): m[t] = m.get(t, 0.0) + float(w)
    return m
def _group_unit(emb, k):
    n, d = emb.shape
    pref = np.zeros((n+1, d), "float64"); np.cumsum(emb, axis=0, out=pref[1:])
    s = pref[k:] - pref[:-k]
    nrm = np.linalg.norm(s, axis=1, keepdims=True); nrm[nrm == 0] = 1.0
    return (s/nrm).astype("float32")
def align_bge(han, vie):
    H, V = len(han), len(vie)
    if not H or not V: return []
    sims = {(dh,dv): _group_unit(h_dense,dh) @ _group_unit(v_dense,dv).T
            for dh,dv in ALIGN["merge_modes"] if dh and dv}
    NEG=-1e9; dp=[[NEG]*(V+1) for _ in range(H+1)]; back=[[None]*(V+1) for _ in range(H+1)]
    dp[0][0]=0.0; thr=ALIGN["sim_threshold"]; modes=ALIGN["merge_modes"]
    for i in tqdm(range(H+1), desc="bge DP align", unit="row"):
        for j in range(V+1):
            base=dp[i][j]
            if base<=NEG: continue
            for dh,dv in modes:
                ni,nj=i+dh,j+dv
                if ni>H or nj>V or (dh==0 and dv==0): continue
                s=float(sims[(dh,dv)][i,j]) if (dh and dv) else 0.0
                gain=s if s>=thr else (s-1.0)
                if base+gain>dp[ni][nj]:
                    dp[ni][nj]=base+gain; back[ni][nj]=(i,j,dh,dv,round(s,3))
    pairs,i,j=[],H,V
    while (i,j)!=(0,0):
        prev=back[i][j]
        if prev is None: break
        pi,pj,dh,dv,s=prev
        am=" ".join(han[k]["am_han_viet"] for k in range(pi,pi+dh))
        vt=" ".join(vie[k]["text"] for k in range(pj,pj+dv))
        lex=round(lexical_sim(am,vt),3)
        bge_sp=round(float(model.compute_lexical_matching_score(
            _merge_sparse(h_sparse,range(pi,pi+dh)),
            _merge_sparse(v_sparse,range(pj,pj+dv)))),3) if (dh and dv) else 0.0
        pairs.append({"han_ids":[han[k]["id"] for k in range(pi,pi+dh)],
            "vie_ids":[vie[k]["id"] for k in range(pj,pj+dv)],
            "sinonom":"".join(han[k]["sinonom"] for k in range(pi,pi+dh)),
            "am_han_viet":am,"vietnamese":vt,"similarity":s,"lexical":lex,"bge_sparse":bge_sp,
            "suspect":bool(s<BGE_SUSPECT_DENSE and lex<ALIGN["suspect_lexical"])})
        i,j=pi,pj
    pairs.reverse(); return pairs

print("BGE model loaded; will process:", VOLS)

In [ ]:
# ── Consensus setup: Qwen2.5-VL arbiter (+ optional specialist), load ONCE, reuse every vol ──
# Cleans Hán OCR before alignment: Qwen-VL re-reads each column image and overrules base
# PaddleOCR where they disagree. Pure vote / re-transliterate logic is in
# pipeline/han_consensus.py; this cell only loads models and crops columns.
# Toggle USE_CONSENSUS=False to align on raw PaddleOCR. Knobs live in config.CONSENSUS.
USE_CONSENSUS = True
import re, time
from PIL import Image
from pipeline import han_consensus as hc
from pipeline.step3_sinonom import load_hanviet

CN = config.CONSENSUS
HANVIET = load_hanviet()
_HAN = re.compile(r"[\u3400-\u9fff\uf900-\ufaff]")

if USE_CONSENSUS:
    import torch
    spec_rec = None
    if CN["use_spec"]:
        from huggingface_hub import hf_hub_download
        SPEC_DIR = f"{MMCY}/spec_rec"; os.makedirs(SPEC_DIR, exist_ok=True)
        for fn in ["inference.json", "inference.pdiparams", "inference.yml"]:
            hf_hub_download(repo_id=CN["spec_repo"], filename=fn, repo_type="space", local_dir=SPEC_DIR)
        from paddleocr import TextRecognition
        spec_rec = TextRecognition(model_name=CN["spec_name"], model_dir=SPEC_DIR)

    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
    from qwen_vl_utils import process_vision_info
    _qkw = dict(torch_dtype=torch.float16, device_map="auto")
    if CN["load_4bit"]:
        _qkw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
    qmodel = Qwen2_5_VLForConditionalGeneration.from_pretrained(CN["qwen_model"], **_qkw).eval()
    qproc  = AutoProcessor.from_pretrained(CN["qwen_model"])
    _PROMPT = ("图中是古籍木刻版的一列繁体汉字，单列竖排。请严格按从上到下的顺序逐字识别，"
               "不得颠倒次序、不得增字或漏字。只输出繁体汉字本身，不要任何解释、标点、拼音、数字或多余文字。")

    _ROT = {"none": None, "cw": Image.ROTATE_270, "ccw": Image.ROTATE_90}
    _pcache = {}
    def _page(pdir, pg):
        key = (pdir, pg)
        if key not in _pcache:
            _pcache[key] = Image.open(f"{pdir}/page_{pg:04d}.png").convert("RGB")
        return _pcache[key]
    def _crop(pdir, b, rot="none"):
        im = _page(pdir, b["page"]); x0, y0, x1, y1 = b["bbox"]; k = CN["crop_inset"]
        c = im.crop((x0 + k, y0 + k, max(x0 + k + 1, x1 - k), max(y0 + k + 1, y1 - k)))
        if CN["upscale"] != 1:
            c = c.resize((c.width * CN["upscale"], c.height * CN["upscale"]), Image.LANCZOS)
        if _ROT.get(rot) is not None:
            c = c.transpose(_ROT[rot])
        return c
    def _spec_read(pdir, b):
        res = spec_rec.predict(np.array(_crop(pdir, b, CN["spec_rotate"])))
        r = res[0] if isinstance(res, list) else res
        txt = r.get("rec_text", "") if isinstance(r, dict) else getattr(r, "rec_text", "")
        return "".join(_HAN.findall(txt or ""))
    @torch.no_grad()
    def _qwen_read(pdir, b):
        pil = _crop(pdir, b); n = len(b.get("sinonom", "") or "")
        prompt = _PROMPT + (f"这一列大约有 {n} 个字。" if n else "")
        msgs = [{"role": "user", "content": [{"type": "image", "image": pil},
                                             {"type": "text", "text": prompt}]}]
        text = qproc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        imgs, vids = process_vision_info(msgs)
        inp = qproc(text=[text], images=imgs, videos=vids, padding=True, return_tensors="pt").to(qmodel.device)
        out = qmodel.generate(**inp, max_new_tokens=CN["max_new_tok"], do_sample=False)
        gen = out[:, inp.input_ids.shape[1]:]
        return "".join(_HAN.findall(qproc.batch_decode(gen, skip_special_tokens=True)[0]))

    def run_consensus(boxes, pdir):
        """Specialist (optional) on all cols + Qwen on the cols worth re-reading;
        returns (consensus_records, review_records, {id: corrected_text})."""
        _pcache.clear()                                   # free previous vol's page images
        for b in boxes:
            b["_spec"] = _spec_read(pdir, b) if spec_rec is not None else ""
        if CN["vlm_mode"] == "full":
            need = boxes
        elif CN["use_spec"]:
            need = [b for b in boxes if hc.fold(b.get("sinonom", "")) != hc.fold(b["_spec"])]
        else:                                             # gate the VLM on base confidence
            g = CN["qwen_conf_gate"]
            need = [b for b in boxes if (b.get("conf") is None or b["conf"] < g)]
        print(f"   Qwen will read {len(need)}/{len(boxes)} cols (vlm_mode={CN['vlm_mode']})")
        t0 = time.time()
        for n, b in enumerate(need):
            b["_qwen"] = _qwen_read(pdir, b)
            if (n + 1) % 100 == 0:
                el = time.time() - t0
                print(f"   qwen {n+1}/{len(need)} ({el:.0f}s, ETA {el/(n+1)*(len(need)-n-1)/60:.0f}m)")
        for b in boxes:
            b.setdefault("_qwen", "")
        recs = [{"id": b["id"], "page": b["page"], "conf": b.get("conf"),
                 "base": b.get("sinonom", ""), "spec": b["_spec"], "qwen": b["_qwen"]} for b in boxes]
        cons, review = hc.build_consensus(recs, vote_mode=CN["vote_mode"])
        corrected = {r["id"]: r["consensus"] for r in cons}
        return cons, review, corrected

    print("consensus ready:", CN["qwen_model"], "| specialist:", bool(spec_rec), "| vlm_mode:", CN["vlm_mode"])
else:
    def run_consensus(boxes, pdir):
        return [], [], {}
    print("USE_CONSENSUS = False -> aligning on raw PaddleOCR")


In [ ]:
# ── Run every volume; push EACH vol's zip to Drive the moment it finishes ──
import shutil, traceback, time, json as _json
!mkdir -p {DRIVE}/output
done, failed = [], []
for idx, VOL in enumerate(VOLS):
    CHAPTER = idx + 1
    print(f"\n{'='*64}\n#### {VOL}  (chapter {CHAPTER})  ####\n{'='*64}")
    t0 = time.time()
    try:
        # ── hand-off from NB1: ONE zip -> unzip onto LOCAL ssd (Drive FUSE corrupts) ──
        IN_DIR = config.OUT_DIR / VOL
        !cp -f {DRIVE}/out_zips/{VOL}.zip {MMCY}/out/
        !cd {MMCY}/out && unzip -qo {VOL}.zip
        assert (IN_DIR / "pages_han").exists(), \
            f"{IN_DIR}/pages_han missing — run NB1 first (writes out_zips/{VOL}.zip)"
        assert (IN_DIR / "vi_sentences.jsonl").exists(), \
            f"{IN_DIR}/vi_sentences.jsonl missing — NB1 step2 must finish first"

        # ── step 3 — Hán OCR (base = PaddleOCR) ──
        !python -m pipeline.step3_sinonom --vol {VOL}
        assert (IN_DIR / "han_sentences.jsonl").exists(), \
            f"step3 produced no han_sentences.jsonl for {VOL}"

        han_boxes = list(read_jsonl(IN_DIR / "han_boxes.jsonl"))
        han_sents = list(read_jsonl(IN_DIR / "han_sentences.jsonl"))
        vie_sents = list(read_jsonl(IN_DIR / "vi_sentences.jsonl"))

        # ── step 3b — consensus clean: Qwen-VL arbiter fixes base BEFORE alignment ──
        if USE_CONSENSUS and han_boxes:
            cons, review, corrected = run_consensus(han_boxes, str(IN_DIR / "pages_han"))
            qrun = sum(1 for b in han_boxes if b.get("_qwen"))          # count before stripping
            han_boxes = hc.apply_consensus(han_boxes, corrected, HANVIET)   # for char-val + Excel
            han_sents = hc.apply_consensus(han_sents, corrected, HANVIET)   # for alignment
            write_jsonl(IN_DIR / "han_boxes.jsonl", han_boxes)
            write_jsonl(IN_DIR / "han_sentences.jsonl", han_sents)
            write_jsonl(IN_DIR / "consensus.jsonl", cons)
            write_jsonl(IN_DIR / "review_queue.jsonl", review)
            mtr = hc.consensus_metrics(cons, review, VOL, CN["vote_mode"], CN["vlm_mode"], qrun)
            (IN_DIR / "consensus_metrics.json").write_text(
                _json.dumps(mtr, ensure_ascii=False, indent=2), encoding="utf-8")
            print(f"consensus: {mtr['cols']} cols | corrected {mtr['corrected_chars']} chars "
                  f"| review {mtr['review_cols']} cols | auto-settle {mtr['auto_settle_char_pct']}%")

        # ── step 4 — bge-m3 alignment on the CORRECTED Hán -> output_bge/<VOL>/ ──
        OUT_DIR = config.ROOT / "output_bge" / VOL; OUT_DIR.mkdir(parents=True, exist_ok=True)
        vie_kept, n_drop = filter_vi_sentences(vie_sents, ALIGN)
        print(f"{len(han_sents)} Hán | {len(vie_sents)} VI -> kept {len(vie_kept)}, dropped {n_drop}")
        h_dense = model.encode([h["sinonom"] for h in han_sents], batch_size=BATCH,
                               return_dense=True, return_sparse=False)["dense_vecs"]
        v_out   = model.encode([v["text"] for v in vie_kept], batch_size=BATCH,
                               return_dense=True, return_sparse=True)
        v_dense, v_sparse = v_out["dense_vecs"], v_out["lexical_weights"]
        h_sparse = model.encode([h["am_han_viet"] for h in han_sents], batch_size=BATCH,
                                return_dense=False, return_sparse=True)["lexical_weights"]
        h_dense = np.asarray(h_dense, "float32"); v_dense = np.asarray(v_dense, "float32")
        print("dense shapes:", h_dense.shape, v_dense.shape)

        pairs = align_bge(han_sents, vie_kept)
        sp = np.array([p["bge_sparse"] for p in pairs if p["vietnamese"] and p["bge_sparse"]>0])
        SPARSE_THR = round(float(np.percentile(sp, 10)), 3) if len(sp) else 0.0
        for p in pairs:
            p["suspect_bge"] = bool(p["similarity"]<BGE_SUSPECT_DENSE and p["bge_sparse"]<SPARSE_THR)
        print(f"{len(pairs)} pairs | suspect={sum(p['suspect'] for p in pairs)} "
              f"| suspect_bge={sum(p['suspect_bge'] for p in pairs)} | SPARSE_THR={SPARSE_THR}")

        # write output_bge/<VOL>/ (jsonl + Excel)
        write_jsonl(OUT_DIR / "alignment.jsonl", pairs)
        df = pd.DataFrame([{
            "Hán IDs": ", ".join(p["han_ids"]), "Việt IDs": ", ".join(p["vie_ids"]),
            "SinoNom": p["sinonom"], "Âm Hán Việt": p["am_han_viet"],
            "Nghĩa thuần Việt": p["vietnamese"], "Dense (bge)": p["similarity"],
            "Lexical (Jaccard)": p["lexical"], "Sparse (bge)": p["bge_sparse"],
            "Nghi lệch (Jaccard)": "x" if p["suspect"] else "",
            "Nghi lệch (bge)": "x" if p.get("suspect_bge") else "",
        } for p in pairs])
        s = config.ID_SCHEMA
        xlsx = OUT_DIR / f"{s['domain']}{s['subdomain']}{s['genre']}_{s['file_id']}_alignment_bge.xlsx"
        with pd.ExcelWriter(xlsx, engine="openpyxl") as xl:
            df.to_excel(xl, sheet_name="sentence_alignment", index=False)
        print("wrote", OUT_DIR / "alignment.jsonl", "and", xlsx.name)

        # ── char-validation/review lanes (on corrected chars) + visual QA overlays ──
        !python -m pipeline.step4_align --vol {VOL}
        !python -m pipeline.draw_boxes --vol {VOL} --align {MMCY}/output_bge/{VOL}/alignment.jsonl

        # ── stage ONE folder, then push THIS vol's zip to Drive now (FUSE-safe) ──
        STAGE = config.ROOT / 'output' / VOL
        if STAGE.exists(): shutil.rmtree(STAGE)
        STAGE.mkdir(parents=True)
        src = config.OUT_DIR / VOL
        bge = config.ROOT / 'output_bge' / VOL
        for d in ['pages_han_boxed','pages_han_aligned','pages_vi_boxed','pages_vi_aligned']:
            if (src/d).exists(): shutil.copytree(src/d, STAGE/d)
        for f in ['han_boxes.jsonl','han_sentences.jsonl','vi_boxes.jsonl','vi_sentences.jsonl',
                  'char_validation.jsonl','review.jsonl','oov_vocab.jsonl','split_manifest.json',
                  'consensus.jsonl','review_queue.jsonl','consensus_metrics.json']:
            if (src/f).exists(): shutil.copy2(src/f, STAGE/f)
        shutil.copy2(bge/'alignment.jsonl', STAGE/'alignment.jsonl')   # BGE, not step4's 0-pairs
        for x in bge.glob('*_alignment_bge.xlsx'): shutil.copy2(x, STAGE/x.name)
        !cd {MMCY}/output && zip -qr {DRIVE}/output/{VOL}.zip {VOL}
        !ls -lh {DRIVE}/output/{VOL}.zip
        print(f"✅ {VOL} DONE in {(time.time()-t0)/60:.1f} min -> Drive output/{VOL}.zip")
        done.append(VOL)
    except Exception as e:
        traceback.print_exc()
        print(f"❌ {VOL} FAILED: {e} — continuing to next vol")
        failed.append(VOL)

print(f"\n{'='*64}\nSUMMARY  done={done}  failed={failed}")
